# CHAOS space record

Notebook to help understand the spatial resolution of CHAOS-8.6

## Preamble

In [6]:
# setting up file path for imports
import sys
import os
from pathlib import Path

cwd = Path(os.getcwd())
print(cwd)
PROJECT_ROOT = cwd.resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

# installing relevant libraries
import chaosmagpy as cp
import numpy as np
import matplotlib.pyplot as plt

# Project paths / utilities
from src.msc_thesis.paths import *
from src.msc_thesis.synSetup import *
from src.msc_thesis.synUtils import *
from src.msc_thesis.synDMD import *

/home/elliott/projects/msc_thesis_project/Last_stretch/Data_prep
['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects', '/home/elliott', '/home/elliott', '/home/elliott/projects', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/projects', '/home/elliott', '/home/elliott', '/home/elliott/projects', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project'

/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages/chaosmagpy/model_utils.py:795: UserWarning: Input coordinates include the poles.
  warnings.warn('Input coordinates include the poles.')


['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/projects', '/home/elliott', '/home/elliott', '/home/elliott/projects', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
C has shape dimensions: (148, 440, 440)


## Spherical harmonic degree

CHAOS-8.6 provides information on magnetic field evolution for spatial detail up to $N_{max}=20$.

As stated in the CHAOS-8 paper, the practical reliable limit for SV at the CMB is up to $N_{max}=15$.

Generally, the gauss coefficient time information becomes more stable with decreasing degree.

In [ ]:
# apply spherical harmonic truncation function

## Physical grid resolution

This can further be translated to a spatial degree-cell resolution limit.

This is useful for understanding grid downsampling limits.

Due to the spherical harmonic upper limit accuracy limitations established above, a downsampling scheme that is capable of representing signals of up to $N_{max}=15$ should be established.

Following the sampling theorem outlined in Driscoll-Healy (1994), a latitude-longitude spacing of 5x5 degrees is the coarsest downsampling choice appropriate.

In [ ]:
# defining this grid:

r_cmb = 3485 # km radius at CMB
r_earth = 6371.2 # km radius at earth surface

# all data uses ndegree^o evenly spaced grid, incl poles
# latitude, colatitude, longitude (degrees):
ndegree = 5
colatitude = np.arange(0, 181, ndegree) 
latitude = colatitude - 90
longitude = np.arange(-180, 180, ndegree)
# theta, phi (radians):
theta = np.deg2rad(colatitude)
phi = np.deg2rad(longitude)

# from this can define number of spatial points, spatial shape etc
num_lat_samples = len(colatitude)
num_lon_samples = len(longitude)
state_shape = (num_lat_samples, num_lon_samples)
num_spatial_points = num_lat_samples * num_lon_samples

# latitude weighting variables
phi_grid, theta_grid = np.meshgrid(phi, theta)
W_theta = np.sin(theta)
W2D = np.sin(theta_grid)
W2D_norm = W2D / np.sum(W2D)


colatitude sampling is [  0   5  10  15  20  25  30  35  40  45  50  55  60  65  70  75  80  85
  90  95 100 105 110 115 120 125 130 135 140 145 150 155 160 165 170 175
 180]
longitude sampling is [-180 -175 -170 -165 -160 -155 -150 -145 -140 -135 -130 -125 -120 -115
 -110 -105 -100  -95  -90  -85  -80  -75  -70  -65  -60  -55  -50  -45
  -40  -35  -30  -25  -20  -15  -10   -5    0    5   10   15   20   25
   30   35   40   45   50   55   60   65   70   75   80   85   90   95
  100  105  110  115  120  125  130  135  140  145  150  155  160  165
  170  175]
Total number of spatial points is 2664


In [ ]:
# examining it plotted 